In [ ]:
import socket
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
from pathlib import Path
from urllib.parse import urlparse
from typing import Optional
import os
import sys
import time
import glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from functools import partial
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
from threading import Thread

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\1_ClipEmbeddingsCreation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def check_internet(host="8.8.8.8", port=53, timeout=3):
    """Check internet connectivity by attempting a TCP connection."""
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except socket.error:
        return False

def wait_for_internet(max_wait=3600):
    """
    Wait until internet connection is available.
    Exponential backoff up to 1hr total wait.
    Returns True if internet is restored, False if timeout reached.
    """
    wait_time = 5  # start with 5 seconds
    total_wait = 0
    was_offline = False

    while not check_internet():
        was_offline = True
        if total_wait >= max_wait:
            print("No internet connection for 1 hour. Exiting...")
            return False
        print(f"[WARNING] No internet connection. Retrying in {wait_time} seconds...")
        time.sleep(wait_time)
        total_wait += wait_time
        wait_time = min(wait_time * 2, 3600)  # grow wait but cap at 1hr

    if was_offline:
        print("Internet connection restored!")
    return True

In [3]:
def fetch_image(index_row_tuple, total_attempts=1, timeout=10, save_dir_path: Optional[str] = None):
    """
    Downloads an image with retries, keeping it simple and fast.

    Args:
        index_row_tuple: (index, row dict with 'url' and 'caption')
        total_attempts: total number of attempts including the first
        timeout: timeout for the request in seconds

    Returns:
        (index, url, caption, PIL.Image or None)
    """
    index, row = index_row_tuple
    image_url = row["url"]
    caption = row["caption"]

    parsed = urlparse(image_url)
    referer = f"{parsed.scheme}://{parsed.hostname}/"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": referer,
        "DNT": "1",
        "Connection": "keep-alive",
    }

    for attempt in range(total_attempts):
        try:
            with requests.Session() as session:
                response = session.get(image_url, timeout=timeout, headers=headers)
                response.raise_for_status()

                image = Image.open(BytesIO(response.content)).convert("RGB")
                image.load()

                if save_dir_path is not None:
                    try:
                        image_path = Path(save_dir_path) / f"{index}.jpg"
                        image.save(image_path)
                    except Exception as e:
                        print(f"Error saving image {index}: {e}")

                return (index, image_url, caption, image)

        except Exception as e:
            # print(f"Error fetching image {index}: {e}")
            if attempt == total_attempts - 1:
                return (index, image_url, caption, None)
            time.sleep(0.5 * (attempt + 1))

In [4]:
def derive_embeddings(batch, processor, model, parquet_file):
    '''
    Function to derive embeddings from a batch of images using a pre-trained model.
    Uses binary batch splitting to isolate bad images instead of falling back to full serial processing.

    Input:
    - batch: A list of tuples, each containing an index, URL, caption, and image object.
    - processor: The processor to prepare the images for the model.
    - model: The pre-trained model to derive embeddings from the images.
    - parquet_file: The name of the Parquet file from which the batch is derived.

    Output:
    - A list of dictionaries containing embedding results or None for failed cases.
    '''

    embedding_results = []
    try:
        indices, urls, captions, images = zip(*batch)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # tqdm.write(f"Processing batch of {len(batch)} images on device: {device}")

        def process_subbatch(sub_indices, sub_urls, sub_captions, sub_images):
            try:
                inputs = processor(images=list(sub_images), return_tensors="pt", padding=True).to(device)
                with torch.no_grad():
                    image_features = model.get_image_features(**inputs)
                image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

                for i in range(len(sub_images)):
                    embedding_results.append({
                        'original_image_index': sub_indices[i],
                        'url': sub_urls[i],
                        'caption': sub_captions[i],
                        'embeddings_result': image_embeddings[i].cpu().numpy(),
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })
            except Exception as e:
                if len(sub_images) == 1:
                    tqdm.write(f"[WARNING] Skipping image at index {sub_indices[0]} due to error: {e}")
                    embedding_results.append({
                        'original_image_index': sub_indices[0],
                        'url': sub_urls[0],
                        'caption': sub_captions[0],
                        'embeddings_result': None,
                        'similarity': None,
                        'parquet_file_name': parquet_file
                    })
                else:
                    mid = len(sub_images) // 2
                    process_subbatch(sub_indices[:mid], sub_urls[:mid], sub_captions[:mid], sub_images[:mid])
                    process_subbatch(sub_indices[mid:], sub_urls[mid:], sub_captions[mid:], sub_images[mid:])

        # Start processing the full batch with fault tolerance
        process_subbatch(indices, urls, captions, images)
        return embedding_results
    except Exception as e:
        print(f"Error processing batch: {e}")
        return embedding_results


In [ ]:
def download_worker(df, batch_size, number_of_workers, save_image_directory, queue):
    """
    Producer: downloads batches and pushes them into the queue.
    Uses wait_for_internet() to pause when offline.
    """
    total_batches = (len(df) + batch_size - 1) // batch_size

    try:
        with tqdm(total=total_batches, desc="Downloading Batches", position=0) as pbar:
            for batch_start in range(0, len(df), batch_size):
                batch_end = min(batch_start + batch_size, len(df))
                batch_rows = list(df.iloc[batch_start:batch_end].iterrows())

                # Wait until internet is available (already handles timeout)
                if not wait_for_internet():
                    print("[WARNING] Internet unavailable for too long. Aborting downloads.")
                    break  # exit producer loop gracefully

                # Download batch in parallel
                with ThreadPoolExecutor(max_workers=number_of_workers) as executor:
                    fetch_with_path = partial(fetch_image, save_dir_path=save_image_directory)
                    fetched = list(executor.map(fetch_with_path, batch_rows))

                valid = [i for i in fetched if i[3] is not None]
                failed = [i for i in fetched if i[3] is None]

                # Push batch to consumer queue
                queue.put((valid, failed, batch_start, batch_end))
                pbar.update(1)  # advance download progress bar

    except Exception as e:
        print(f"[WARNING] Download worker stopped: {e}")

    finally:
        # Always send poison pill so consumer exits
        queue.put(None)

def process_parquet_images(
    directory,
    parquet_file,
    output_base_dir,
    model_name,
    batch_size=200,
    number_of_workers=200,
    image_limit=3_000_000,
    save_interval=5000,
    dataset_split_size=2_000_000,
    keep_n_recent_saves=2,
    row_grp_size=300000,
    filter_keyword="",
    save_image_directory=None
):
    # Prepare output directory
    output_subdir_name = (
        f"{filter_keyword.replace(' ', '_')}_{model_name.replace('/', '_').replace('-', '_')}"
        if filter_keyword else
        f"all_images_{model_name.replace('/', '_').replace('-', '_')}"
    )
    dataset_dir = os.path.join(output_base_dir, output_subdir_name, parquet_file.replace(".parquet", "_embeddings"))
    os.makedirs(dataset_dir, exist_ok=True)

    # Load source parquet file
    parquet_path = os.path.join(directory, parquet_file)
    print(f"Loading DataFrame from {parquet_path}...")

    try:
        df = pd.read_parquet(
            parquet_path,
            columns=["url", "caption"]
        )
        print(f"Loaded {len(df)} rows.")
    except FileNotFoundError:
        raise FileNotFoundError(f"Input file not found at {parquet_path}")


    if filter_keyword:
        df = df[df["caption"].str.contains(filter_keyword, case=False, na=False)].copy()
    df = df.head(image_limit)

    # Initialize model
    processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # Resume tracking
    processed_indices = set()
    part_files = sorted(glob.glob(os.path.join(dataset_dir, "part-*.parquet")))

    finalized_parts = []
    for f in part_files:
        if "-checkpoint-" in f:
            continue
        try:
            table = pq.read_table(f, columns=["original_image_index"])
            processed_indices.update(table["original_image_index"].to_numpy())
            part_num = int(os.path.basename(f).split("-")[1].split(".")[0])
            finalized_parts.append(part_num)
        except:
            pass

    max_finalized_index = max(finalized_parts) + 1 if finalized_parts else 0

    # Resume from latest checkpoint if exists
    checkpoint_files = sorted(
        glob.glob(os.path.join(dataset_dir, "part-*-checkpoint-*.parquet")),
        key=os.path.getmtime,
        reverse=True
    )

    checkpoint_df = pd.DataFrame()
    checkpoint_part_index = None
    if checkpoint_files:
        latest_checkpoint = checkpoint_files[0]
        base_name = os.path.basename(latest_checkpoint)
        parts = base_name.split("-")
        try:
            checkpoint_part_index = int(parts[1])
        except:
            checkpoint_part_index = None
        try:
            table = pq.read_table(latest_checkpoint)
            checkpoint_df = table.to_pandas()
            print(f"Resuming from checkpoint part-{checkpoint_part_index:05d} with {len(checkpoint_df)} rows")
        except:
            print("[WARNING] Failed to load checkpoint, starting fresh.")

    # Attempt to resume from incomplete part if no checkpoint
    if checkpoint_part_index is None and finalized_parts:
        last_part_num = max(finalized_parts)
        last_part_path = os.path.join(dataset_dir, f"part-{last_part_num:05d}.parquet")
        try:
            table = pq.read_table(last_part_path)
            last_part_df = table.to_pandas()
            if len(last_part_df) < dataset_split_size:
                checkpoint_df = last_part_df
                checkpoint_part_index = last_part_num
                split_index = last_part_num
                processed_indices.update(last_part_df["original_image_index"].to_numpy())
                print(f"Resuming incomplete part-{split_index:05d} with {len(checkpoint_df)} rows")
            else:
                split_index = last_part_num + 1
        except:
            split_index = max_finalized_index
    else:
        split_index = checkpoint_part_index if checkpoint_part_index is not None else max_finalized_index

    # Filter unprocessed rows
    df = df[~df.index.isin(processed_indices)]
    if not checkpoint_df.empty:
        checkpoint_indices = set(checkpoint_df["original_image_index"])
        df = df[~df.index.isin(checkpoint_indices)]

    print(f"Remaining to process after excluding checkpoint data: {len(df)}")

    buffer = []
    total_processed = 0

    # --- Producer–Consumer Setup ---
    download_queue = Queue(maxsize=10)  # prevent memory blow-up
    dl_thread = Thread(
        target=download_worker,
        args=(df, batch_size, number_of_workers, save_image_directory, download_queue),
        daemon=True
    )
    dl_thread.start()

    batch_num = 0
    total_batches = (len(df) + batch_size - 1) // batch_size

    with tqdm(total=total_batches, desc="Processing Batches") as pbar:
        while True:
            item = download_queue.get()
            if item is None:  # poison pill
                break

            valid, failed, batch_start, batch_end = item
            batch_num += 1

            print(f"Batch {batch_num}: Downloaded {len(valid)} images, {len(failed)} failed.")

            # --- Embedding step ---
            batch_embedding_st_time = time.time()
            results = derive_embeddings(valid, processor, model, parquet_file)
            batch_embedding_elapsed_time = time.time() - batch_embedding_st_time

            for f in failed:
                results.append({
                    'original_image_index': f[0],
                    'url': f[1],
                    'caption': f[2],
                    'embeddings_result': None,
                    'similarity': None,
                    'parquet_file_name': parquet_file
                })

            print(
                f"Batch {batch_num} processed in: "
                f"Embedding {batch_embedding_elapsed_time:.2f}s | "
                f"Total {batch_embedding_elapsed_time:.2f}s - "
                f"Success Links: {len(valid)} & Failed Links: {len(failed)}"
            )

            buffer.extend(results)
            total_processed += len(results)

            # --- Checkpoint save ---
            if total_processed % save_interval < batch_size or batch_end == len(df):
                checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame(buffer)], ignore_index=True)
                checkpoint_df.drop_duplicates(subset=['original_image_index'], keep='last', inplace=True)
                checkpoint_df.sort_values(by='original_image_index', inplace=True)
                checkpoint_df.reset_index(drop=True, inplace=True)
                checkpoint_path = os.path.join(
                    dataset_dir,
                    f"part-{split_index:05d}-checkpoint-{len(checkpoint_df)}.parquet"
                )
                pq.write_table(pa.Table.from_pandas(checkpoint_df), checkpoint_path, row_group_size=row_grp_size)
                tqdm.write(f"Checkpoint saved: {checkpoint_path}")
                buffer = []

                # Clean up old checkpoints
                all_checkpoints = sorted(
                    glob.glob(os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")),
                    key=os.path.getmtime,
                    reverse=True
                )
                for old_cp in all_checkpoints[keep_n_recent_saves:]:
                    try:
                        os.remove(old_cp)
                        tqdm.write(f"Deleted old checkpoint: {old_cp}")
                    except Exception as e:
                        tqdm.write(f"Could not delete checkpoint {old_cp}: {e}")

            # --- Finalize part ---
            if len(checkpoint_df) >= dataset_split_size:
                final_path = os.path.join(dataset_dir, f"part-{split_index:05d}.parquet")
                pq.write_table(pa.Table.from_pandas(checkpoint_df), final_path, row_group_size=row_grp_size)
                tqdm.write(f"Finalized: {final_path}")

                # Clean up checkpoints for this part
                checkpoint_glob = os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")
                for cp_file in glob.glob(checkpoint_glob):
                    try:
                        os.remove(cp_file)
                        tqdm.write(f"Removed checkpoint: {cp_file}")
                    except Exception as e:
                        tqdm.write(f"Could not remove checkpoint {cp_file}: {e}")

                checkpoint_df = pd.DataFrame()
                split_index += 1

            pbar.update(1)  # advance progress bar

    # --- Final flush ---
    if not checkpoint_df.empty:
        final_path = os.path.join(dataset_dir, f"part-{split_index:05d}.parquet")
        pq.write_table(pa.Table.from_pandas(checkpoint_df), final_path, row_group_size=row_grp_size)
        tqdm.write(f"Completed: {final_path}")

        checkpoint_glob = os.path.join(dataset_dir, f"part-{split_index:05d}-checkpoint-*.parquet")
        for cp_file in glob.glob(checkpoint_glob):
            try:
                tqdm.write(f"Removed checkpoint: {cp_file}")
                os.remove(cp_file)
            except Exception as e:
                tqdm.write(f"Could not remove checkpoint {cp_file}: {e}")


In [ ]:
# --- Configuration ---
filter_keyword = ""
directory = r"F:\Thesis\RE-LAION-5B_Dataset\relaion2B-en-research-safe"

output_base_dir = r"../clip_embeddings_resumable_symlink"
output_image_base_dir = r"F:\Thesis"

clip_model_names = [
    "openai/clip-vit-base-patch16",
    "openai/clip-vit-base-patch32",
    "openai/clip-vit-large-patch14"
]

model_index = 2
model_name = clip_model_names[model_index]

batch_size = 200
number_of_workers = 100
save_interval = 5000
row_grp_size = 200_000
keep_n_recent_saves = 3
image_limit = 17_388_200

# -------------------------------------------------------
# Discover all parquet files
# -------------------------------------------------------

parquet_dir = Path(directory)

parquet_files = sorted(
    parquet_dir.glob("*.parquet"),
    key=lambda x: int(x.stem) if x.stem.isdigit() else x.stem
)

if not parquet_files:
    raise ValueError(f"No parquet files found in {directory}")

print(f"Found {len(parquet_files)} parquet files.")


# -------------------------------------------------------
# Process each parquet file sequentially
# -------------------------------------------------------

for idx, parquet_path in enumerate(parquet_files, start=1):

    parquet_stem = parquet_path.stem
    if parquet_stem.startswith("part_"):
        parquet_stem = parquet_stem[len("part_"):]
    save_image_directory = os.path.join(output_image_base_dir, f"{parquet_stem}_images")

    os.makedirs(save_image_directory, exist_ok=True)
    os.chmod(save_image_directory, 0o755)

    print("\n" + "=" * 80)
    print(f"Processing file {idx}/{len(parquet_files)}: {parquet_path.name}")
    print(f"Saving images to: {save_image_directory}")
    print("=" * 80 + "\n")

    process_parquet_images(
        directory=directory,
        parquet_file=parquet_path.name,
        output_base_dir=output_base_dir,
        model_name=model_name,
        batch_size=batch_size,
        number_of_workers=number_of_workers,
        image_limit=image_limit,
        save_interval=save_interval,
        row_grp_size=row_grp_size,
        keep_n_recent_saves=keep_n_recent_saves,
        filter_keyword=filter_keyword,
        dataset_split_size=2_000_000,
        save_image_directory=save_image_directory
    )